In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import json
import sys
from datasets import load_dataset

In [2]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()
DATA_DIR = ROOT / "data"
CORPUS_PATH = DATA_DIR / "corpus" / "pooled_corpus.parquet"
SPLITS_DIR = ROOT / "splits"

# Folders end in _DIR, files end in _PATH
GOLD_DIR = ROOT / "results" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
GOLD_PATH = GOLD_DIR / "pilot_gold.json"

# Scratch prediction files from the tests go in data/tmp (git-ignored), not results/
TMP_DIR = DATA_DIR / "tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = TMP_DIR / "pilot_gold_as_pred.json"
CORRUPTED_PATH = TMP_DIR / "pilot_corrupted_pred.json"

# Import the evaluation module the way later stages will: repo root on sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from eval.evaluation import (
    gold_converter, write_predictions, run_official_eval,
    score_answers, score_retrieval, exact_match_score, f1_score,
)

In [3]:
# Get Pilot Slice ids for development
with open(SPLITS_DIR / "pilot.json", "r", encoding="utf-8") as file:
    slice_ids = json.load(file)

# Load Dataset for questions
hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

# Get questions for the slice ids only,
train_q = hotpot_ds["train"].select_columns(["id", "answer", "supporting_facts"]).to_pandas()
train_q = train_q.set_index("id")
slice_q = train_q.loc[slice_ids]
assert list(slice_q.index) == slice_ids


In [4]:
gold = gold_converter(slice_q)

assert len(gold) == 200
assert [g["_id"] for g in gold] == slice_ids
print (f"{gold[0]} --- {slice_q.iloc[0]}")

with open(GOLD_PATH, "w") as f:
    json.dump(gold, f, indent=2)

{'_id': '5abd94d55542992ac4f382d5', 'answer': 'Steve Jobs', 'supporting_facts': [['Susan Barnes (computing)', 2], ['Steve Jobs', 0]]} --- answer                                                     Steve Jobs
supporting_facts    {'title': ['Susan Barnes (computing)', 'Steve ...
Name: 5abd94d55542992ac4f382d5, dtype: object


## Convert Outputs
- Turn system outputs into the file the official scripts reads.

**Inputs**: 
- answers: {qid: answer}
- sp [optional]: {qid: [[title, sent_id], ...]} $\rightarrow$ used only if we predict supporting facts
- expected_ids: The IDs that should be there
- output path

In [5]:
# Prediction writer tests
gold_answers = {g["_id"]: g["answer"] for g in gold}
gold_sp = {g["_id"]: g["supporting_facts"] for g in gold}

# Test 1: gold answers + gold sp -> every metric 1.0
write_predictions(gold_answers, slice_ids, PRED_PATH, sp=gold_sp)
metrics, printed = run_official_eval(PRED_PATH, GOLD_PATH)
assert all(v == 1.0 for v in metrics.values()), metrics
print(metrics)

# Test 2: gold answers, no sp -> em = f1 = 1.0, sp scores 0, no "missing" lines
write_predictions(gold_answers, slice_ids, PRED_PATH)
metrics, printed = run_official_eval(PRED_PATH, GOLD_PATH)
assert metrics["em"] == metrics["f1"] == 1.0 and metrics["sp_f1"] == 0.0, metrics
assert not any("missing" in line for line in printed), printed[:3]

# Test 3: one ID missing -> ValueError before writing
try:
    write_predictions({k: v for k, v in gold_answers.items() if k != slice_ids[0]}, slice_ids, PRED_PATH)
    raise AssertionError("missing ID was not caught")
except ValueError as e:
    print("Test 3 caught:", e)

# Test 4: a None answer -> TypeError
try:
    write_predictions({**gold_answers, slice_ids[0]: None}, slice_ids, PRED_PATH)
    raise AssertionError("None answer was not caught")
except TypeError as e:
    print("Test 4 caught:", e)

print("All write_predictions tests passed")

{'em': 1.0, 'f1': 1.0, 'prec': 1.0, 'recall': 1.0, 'sp_em': 1.0, 'sp_f1': 1.0, 'sp_prec': 1.0, 'sp_recall': 1.0, 'joint_em': 1.0, 'joint_f1': 1.0, 'joint_prec': 1.0, 'joint_recall': 1.0}
Test 3 caught: 1 missing, 0 extra; e.g. missing ['5abd94d55542992ac4f382d5'], extra []
Test 4 caught: 1 answers are not strings, e.g. [('5abd94d55542992ac4f382d5', 'NoneType')]
All write_predictions tests passed


In [6]:
# qid -> question type (bridge or comparison); a question attribute, not a label
train_types = hotpot_ds["train"].select_columns(["id", "type"]).to_pandas()
qid_to_type = dict(zip(train_types.id, train_types.type))

In [7]:
# Answer scorer tests

# Test 1: gold as predictions -> EM = F1 = 100 overall and per type
gold_preds = write_predictions(gold_answers, slice_ids, PRED_PATH)
per_question, summary = score_answers(gold_preds, gold, qid_to_type)
assert all(s["em"] == s["f1"] == 100.0 for s in summary.values()), summary
print(summary)

# Test 2: known cases of the official normalisation
assert exact_match_score("The Beatles", "Beatles")              # articles removed
assert f1_score("yes", "no") == (0, 0, 0)                        # yes/no special case
f1, precision, recall = f1_score("Steve", "Steve Jobs")          # partial answer
assert exact_match_score("Steve", "Steve Jobs") is False and abs(f1 - 2 / 3) < 1e-9

# Test 3: corrupted predictions -> same EM / F1 as the official script
# (a third correct, a third cut to the first word, a third wrong)
corrupted = {}
for i, g in enumerate(gold):
    answer = g["answer"]
    corrupted[g["_id"]] = [answer, answer.split()[0], "wrong answer"][i % 3]
corrupted_preds = write_predictions(corrupted, slice_ids, CORRUPTED_PATH)

official, _ = run_official_eval(CORRUPTED_PATH, GOLD_PATH)
per_question, summary = score_answers(corrupted_preds, gold, qid_to_type)
assert abs(official["em"] - summary["overall"]["em"] / 100) < 1e-9, (official["em"], summary["overall"])
assert abs(official["f1"] - summary["overall"]["f1"] / 100) < 1e-9, (official["f1"], summary["overall"])
print(f"Official: EM {official['em']:.4f}, F1 {official['f1']:.4f} | ours: {summary['overall']}")

print("All score_answers tests passed")

{'overall': {'n': 200, 'em': 100.0, 'f1': 100.0}, 'bridge': {'n': 160, 'em': 100.0, 'f1': 100.0}, 'comparison': {'n': 40, 'em': 100.0, 'f1': 100.0}}


Official: EM 0.4300, F1 0.5458 | ours: {'n': 200, 'em': 43.0, 'f1': 54.57777777777778}
All score_answers tests passed


In [8]:
# pid -> title, for scoring retrieval runs by title
corpus_titles = pd.read_parquet(CORPUS_PATH, columns=["pid", "title"])
pid_to_title = dict(zip(corpus_titles.pid, corpus_titles.title))

In [9]:
# Retrieval scorer tests

# title -> pids, to place gold paragraphs at chosen ranks
title_to_pids = corpus_titles.groupby("title").pid.apply(list).to_dict()
gold_titles_of = {g["_id"]: sorted({t for t, _ in g["supporting_facts"]}) for g in gold}
first_gold = gold[0]
qid0 = first_gold["_id"]
title_a, title_b = gold_titles_of[qid0]
filler_pids = [pid for pid, title in zip(corpus_titles.pid, corpus_titles.title)
               if title not in (title_a, title_b)][:3]

# Test 1: gold pids ranked first -> both@2 = any@2 = 100, mean gold@2 = 2
gold_run = {qid: [title_to_pids[t][0] for t in titles] for qid, titles in gold_titles_of.items()}
_, summary = score_retrieval(gold_run, gold, qid_to_type, pid_to_title)
assert summary["overall"]["both@2"] == summary["overall"]["any@2"] == 100.0
assert summary["overall"]["mean_gold@2"] == 2.0

# Test 2: empty lists -> everything 0, no crash
_, summary = score_retrieval({qid: [] for qid in gold_titles_of}, gold, qid_to_type, pid_to_title)
assert all(v == 0.0 for k, v in summary["overall"].items() if k != "n")

# Test 3: gold at ranks 1 and 5 -> both@2 = 0, any@2 = 1, both@5 = 1, gold@5 = 2
run = {qid0: [title_to_pids[title_a][0]] + filler_pids + [title_to_pids[title_b][0]]}
per_question, _ = score_retrieval(run, [first_gold], qid_to_type, pid_to_title)
row = per_question.iloc[0]
assert (row.rank_gold_1, row.rank_gold_2) == (1, 5)
assert (row["both@2"], row["any@2"], row["both@5"], row["gold@5"]) == (0.0, 1.0, 1.0, 2)

# Test 4: both versions of a duplicated title take one slot, not two
duplicate_pids = next(pids for pids in title_to_pids.values() if len(pids) == 2)
run = {qid0: duplicate_pids + [title_to_pids[title_a][0], title_to_pids[title_b][0]]}
per_question, _ = score_retrieval(run, [first_gold], qid_to_type, pid_to_title)
row = per_question.iloc[0]
assert row.n_retrieved == 3 and (row.rank_gold_1, row.rank_gold_2) == (2, 3)

# Test 5: a question missing from the run -> ValueError
try:
    score_retrieval({}, gold, qid_to_type, pid_to_title)
    raise AssertionError("missing questions were not caught")
except ValueError as e:
    print("Test 5 caught:", e)

print("All score_retrieval tests passed")

Test 5 caught: 200 gold questions missing from the run, e.g. ['5abd94d55542992ac4f382d5', '5a8afca455429971feec45ca', '5a88a2045542997e5c09a64f', '5abab0b955429955dce3eece', '5a8451b355429933447460d6']
All score_retrieval tests passed
